In [2]:


import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM
from tensorflow.keras.utils import to_categorical



df = pd.read_csv(r"C:\Users\MrLaptop\Desktop\TB_Chest_Radiography_Database\Mental Health Dataset.csv")

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns)



df = df.drop(columns=["Timestamp"], errors="ignore")
df = df.fillna("Unknown")

target_col = "treatment"

feature_cols = [
    "Gender",
    "Country",
    "Occupation",
    "self_employed",
    "family_history",
    "Days_Indoors",
    "Growing_Stress",
    "Changes_Habits",
    "Mental_Health_History",
    "Mood_Swings",
    "Coping_Struggles",
    "Work_Interest",
    "Social_Weakness",
    "mental_health_interview",
    "care_options"
]

X = df[feature_cols].copy()
y = df[target_col].copy()


encoders = {}

for col in feature_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    encoders[col] = le

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y.astype(str))

num_classes = len(np.unique(y))

print("Target classes:", target_encoder.classes_)



X_train, X_test, y_train, y_test = train_test_split(
    X.values,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)



ann_model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

ann_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining ANN model...")

ann_history = ann_model.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    verbose=1
)

ann_loss, ann_acc = ann_model.evaluate(X_test, y_test_cat, verbose=0)
print("\nANN Test Accuracy:", ann_acc)

ann_pred = np.argmax(ann_model.predict(X_test), axis=1)

print("\nANN Classification Report:")
print(classification_report(y_test, ann_pred, target_names=target_encoder.classes_))



max_feature_value = int(X.values.max()) + 1

lstm_model = Sequential([
    Embedding(
        input_dim=max_feature_value,
        output_dim=32,
        input_length=X_train.shape[1]
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(num_classes, activation="softmax")
])

lstm_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nTraining LSTM model...")

lstm_history = lstm_model.fit(
    X_train,
    y_train_cat,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    verbose=1
)

lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test_cat, verbose=0)
print("\nLSTM Test Accuracy:", lstm_acc)

lstm_pred = np.argmax(lstm_model.predict(X_test), axis=1)

print("\nLSTM Classification Report:")
print(classification_report(y_test, lstm_pred, target_names=target_encoder.classes_))



def safe_encode(col, value):
    value = str(value).strip()

    if value in encoders[col].classes_:
        return encoders[col].transform([value])[0]
    else:
        return 0



def mental_health_chatbot():
    print("\n======================================")
    print("AI Mental Health Detection Chatbot")
    print("======================================")
    print("Answer using: Yes / No / Maybe")
    print("This chatbot is for educational use only.")
    print("It is not a medical diagnosis.\n")

    user_data = {}

    # Only depression / mental-health related questions
    q1 = input("Do you feel your stress is increasing recently? ")
    q2 = input("Have your sleeping or eating habits changed recently? ")
    q3 = input("Do you have a previous mental health history? ")
    q4 = input("Do you experience frequent mood swings? ")
    q5 = input("Do you struggle to cope with daily problems? ")
    q6 = input("Have you lost interest in work, study, or daily activities? ")
    q7 = input("Do you feel socially weak, isolated, or withdrawn? ")
    q8 = input("Do you feel uncomfortable talking about your mental health? ")
    q9 = input("Does your family have a history of mental health problems? ")

    # Map chatbot questions to dataset columns
    user_data["Growing_Stress"] = q1
    user_data["Changes_Habits"] = q2
    user_data["Mental_Health_History"] = q3
    user_data["Mood_Swings"] = q4
    user_data["Coping_Struggles"] = q5
    user_data["Work_Interest"] = q6
    user_data["Social_Weakness"] = q7
    user_data["mental_health_interview"] = q8
    user_data["family_history"] = q9

    # Default values for non-chatbot dataset columns
    user_data["Gender"] = "Male"
    user_data["Country"] = "Unknown"
    user_data["Occupation"] = "Student"
    user_data["self_employed"] = "No"
    user_data["Days_Indoors"] = "1-14 days"
    user_data["care_options"] = "No"

    # Convert Maybe to dataset-friendly value
    for key in user_data:
        if str(user_data[key]).lower() == "maybe":
            user_data[key] = "Maybe"

    encoded_input = []

    for col in feature_cols:
        encoded_value = safe_encode(col, user_data[col])
        encoded_input.append(encoded_value)

    user_array = np.array(encoded_input).reshape(1, -1)

    # ANN prediction
    ann_prob = ann_model.predict(user_array)
    ann_class = np.argmax(ann_prob, axis=1)[0]
    ann_result = target_encoder.inverse_transform([ann_class])[0]

    # LSTM prediction
    lstm_prob = lstm_model.predict(user_array)
    lstm_class = np.argmax(lstm_prob, axis=1)[0]
    lstm_result = target_encoder.inverse_transform([lstm_class])[0]

    # Final decision
    if ann_result == "Yes" or lstm_result == "Yes":
        final_result = "Yes"
    else:
        final_result = "No"

    print("\n======================================")
    print("Chatbot Result")
    print("======================================")

    print("ANN Prediction:", ann_result)
    print("LSTM Prediction:", lstm_result)

    if final_result == "Yes":
        print("""
Chatbot Response:
Based on your answers, the system suggests that you may be experiencing signs of mental stress or depression-related difficulty.

This is not a medical diagnosis, but it may be helpful to talk to a counselor, psychologist, doctor, or trusted person.

If you feel hopeless, unsafe, or have thoughts of harming yourself, please contact emergency help or someone close to you immediately.
""")
    else:
        print("""
Chatbot Response:
Based on your answers, the system does not strongly indicate a need for mental health treatment.

However, if you continue to feel low, stressed, anxious, isolated, or emotionally unstable, it is still a good idea to talk to someone you trust or seek professional advice.
""")



mental_health_chatbot()

Dataset shape: (292364, 17)
         Timestamp  Gender        Country Occupation self_employed  \
0  8/27/2014 11:29  Female  United States  Corporate           NaN   
1  8/27/2014 11:31  Female  United States  Corporate           NaN   
2  8/27/2014 11:32  Female  United States  Corporate           NaN   
3  8/27/2014 11:37  Female  United States  Corporate            No   
4  8/27/2014 11:43  Female  United States  Corporate            No   

  family_history treatment Days_Indoors Growing_Stress Changes_Habits  \
0             No       Yes    1-14 days            Yes             No   
1            Yes       Yes    1-14 days            Yes             No   
2            Yes       Yes    1-14 days            Yes             No   
3            Yes       Yes    1-14 days            Yes             No   
4            Yes       Yes    1-14 days            Yes             No   

  Mental_Health_History Mood_Swings Coping_Struggles Work_Interest  \
0                   Yes      Medium       

C:\Users\MrLaptop\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


731/731 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5588 - loss: 0.7672 - val_accuracy: 0.6777 - val_loss: 0.5945
Epoch 2/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6920 - loss: 0.5948 - val_accuracy: 0.7116 - val_loss: 0.5813
Epoch 3/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7074 - loss: 0.5747 - val_accuracy: 0.7225 - val_loss: 0.5584
Epoch 4/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7180 - loss: 0.5598 - val_accuracy: 0.7347 - val_loss: 0.5466
Epoch 5/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7284 - loss: 0.5460 - val_accuracy: 0.7375 - val_loss: 0.5306
Epoch 6/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7328 - loss: 0.5361 - val_accuracy: 0.7445 - val_loss: 0.5185
Epoch 7/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7382 - loss: 0.5260 - val_accuracy: 0.7430 - val_loss: 0.5119
Epoch 8/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7430 - loss: 0.5171 - val_accuracy: 0.7504 - val_

C:\Users\MrLaptop\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


731/731 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.6468 - loss: 0.6264 - val_accuracy: 0.7063 - val_loss: 0.5695
Epoch 2/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.7178 - loss: 0.5506 - val_accuracy: 0.7315 - val_loss: 0.5210
Epoch 3/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.7318 - loss: 0.5222 - val_accuracy: 0.7475 - val_loss: 0.4962
Epoch 4/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7494 - loss: 0.4959 - val_accuracy: 0.7702 - val_loss: 0.4713
Epoch 5/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.7667 - loss: 0.4690 - val_accuracy: 0.7734 - val_loss: 0.4507
Epoch 6/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.7755 - loss: 0.4514 - val_accuracy: 0.7800 - val_loss: 0.4401
Epoch 7/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.7788 - loss: 0.4425 - val_accuracy: 0.7861 - val_loss: 0.4333
Epoch 8/10
731/731 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7819 - loss: 0.4362 - val_accurac

Do you feel your stress is increasing recently?  yes
Have your sleeping or eating habits changed recently?  yes
Do you have a previous mental health history?  no
Do you experience frequent mood swings?  yes
Do you struggle to cope with daily problems?  yes
Have you lost interest in work, study, or daily activities?  yes
Do you feel socially weak, isolated, or withdrawn?  yes
Do you feel uncomfortable talking about your mental health?  no
Does your family have a history of mental health problems?  no


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

Chatbot Result
ANN Prediction: No
LSTM Prediction: No

Chatbot Response:
Based on your answers, the system does not strongly indicate a need for mental health treatment.

However, if you continue to feel low, stressed, anxious, isolated, or emotionally unstable, it is still a good idea to talk to someone you trust or seek professional advice.

